# Ventas de cafetería
Este análisis explora las ventas de tres sucursales de cafetería 
(Astoria, Hell's Kitchen y Lower Manhattan) para entender qué 
productos mueven el negocio, cómo se comportan las sucursales 
entre sí y qué patrones temporales existen.

**Pregunta central:** ¿Las tres sucursales se comportan igual, 
o hay diferencias en preferencias de productos y rendimiento?

# Librerías
Módulos necesarios para el análisis.

In [12]:
import plotly.io as pio
pio.renderers.default = "notebook"
'''
Para importar el cuaderno como html, escribe el siguiente comando en la terminal:
jupyter nbconvert --to hmtl notebook.ipynb
'''

'\nPara importar el cuaderno como html, escribe el siguiente comando en la terminal:\njupyter nbconvert --to hmtl notebook.ipynb\n'

In [13]:
# básicas
import pandas as pd
import numpy as np
import sys
import kaleido # para exportar gráficas
# visualización
import matplotlib.pyplot as plt
import seaborn as sns 
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print('Python version', sys.version)
print('Pandas version', pd.__version__)
print('✅ Librerías importadas con éxito.')

Python version 3.10.13 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:15:57) [MSC v.1916 64 bit (AMD64)]
Pandas version 2.3.2
✅ Librerías importadas con éxito.


# Datos

In [14]:
df=pd.read_csv(r'../data/coffee_shop_sales.csv')
print('✅ Datos importados con éxito')
print('🔍\n::INFORMACIÓN GENERAL::\n')
print(df.info())

✅ Datos importados con éxito
🔍
::INFORMACIÓN GENERAL::

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149116 entries, 0 to 149115
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_id    149116 non-null  int64  
 1   transaction_date  149116 non-null  object 
 2   transaction_time  149116 non-null  object 
 3   transaction_qty   149116 non-null  int64  
 4   store_id          149116 non-null  int64  
 5   store_location    149116 non-null  object 
 6   product_id        149116 non-null  int64  
 7   unit_price        149116 non-null  float64
 8   product_category  149116 non-null  object 
 9   product_type      149116 non-null  object 
 10  product_detail    149116 non-null  object 
dtypes: float64(1), int64(4), object(6)
memory usage: 12.5+ MB
None


In [15]:
df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,01/01/2023,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,01/01/2023,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,01/01/2023,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,01/01/2023,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,01/01/2023,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


Se puede identificar las siguientes variables categóricas
- `store_id`: Identificador para cada establecimiento, se define con un número entero.
- `store_location`: Lugar donde se encuentra el establecimiento, se define con el nombre de la locación.
- `product_id`: Identificador del producto, se define con un número entero.
- `unit_price`: Precio unitario de cada artículo.
- `product_category`: Categoría a la que pertenece cada producto del menú.
- `product_type`: Descripción breve del producto.
- `product_detail`: Descripción más detallada del producto

## Preparación de los datos

In [16]:
# verificamos si hay valores nulos 
df_null=df.isnull().sum().reset_index()

if(df_null.iloc[:,1]!=0).any():
    print('⚠️ Advertencia, hay valores nulos')
    print(f'::::::\nValores nulos por columna:\n{df.isna().sum()}')
    print(df_null)
else:
    print('✅ No hay valores nulos')


✅ No hay valores nulos


Para un mejor manejo de los datos temporales se hará la fusión de los datos correspondientes a la hora y fecha en una sola columna.

In [17]:
# conversión a tipo datetime
df['datetime']=pd.to_datetime(
    df['transaction_date'] +' '+ df['transaction_time'],
    format='%d/%m/%Y %H:%M:%S'
)
# ordenamos por fecha
df=df.sort_values('datetime')

# eliminamos las columnas
df=df.drop(columns=['transaction_date', 'transaction_time'])

In [18]:
# se extraen los datos principales de la fecha
df['hour']=df['datetime'].dt.hour
df['day']=df['datetime'].dt.day
df['day_name']=df['datetime'].dt.day_name(locale='es_MX')
df['month_name']=df['datetime'].dt.month_name(locale='es_MX')
df['month']=df['datetime'].dt.month


In [19]:
# agregamos la columna de venta total por registro
df['total']=df['transaction_qty']*df['unit_price']

In [20]:
df.head()

,transaction_id,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail,datetime,hour,day,day_name,month_name,month,total
0,1,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg,2023-01-01 07:06:11,7,1,Domingo,Enero,1,6.0
1,2,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,2023-01-01 07:08:56,7,1,Domingo,Enero,1,6.2
2,3,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg,2023-01-01 07:14:04,7,1,Domingo,Enero,1,9.0
3,4,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm,2023-01-01 07:20:24,7,1,Domingo,Enero,1,2.0
4,5,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,2023-01-01 07:22:41,7,1,Domingo,Enero,1,6.2


# Análisis exploratorio de datos

## Visión general
Antes de explorar el comportamiento de cada sucursal, conviene entender el panorama completo, categorías, tipos y productos venden mas en volumen, cuáles generan más ingresos y ver si coinciden.

In [21]:
print(f'La tabla contiene :\n-> {df.shape[0]:,} registros\n-> {df.shape[1]} columnas')
# Ventas totales
total_ventas=df['total'].sum()
print(f'Total de ventas -> $ {total_ventas:,.2f}')
# Promedio de ventas
promedio_ventas=df['total'].mean()
print(f'Promedio de venta general -> ${promedio_ventas:.2f}')

# visión temporal
inicio=df['datetime'].iloc[0]
fin=df['datetime'].iloc[-1]
print('\nEl rango de fechas (YYYY/mm/dd) de la base de datos es de:')
print(f'Desde: {inicio}')
print(f'Hasta: {fin}')

La tabla contiene :
-> 149,116 registros
-> 16 columnas
Total de ventas -> $ 698,812.33
Promedio de venta general -> $4.69

El rango de fechas (YYYY/mm/dd) de la base de datos es de:
Desde: 2023-01-01 07:06:11
Hasta: 2023-06-30 20:57:19


### Búsqueda de registros atípicos

In [ ]:
# gráfico de cajas y bigotes para identificar posibles registros atípicos
fig=px.box(
    data_frame=df,
    y='total',
    log_y=True,
    title='Diagrama de caja de los ingresos totales.',
    subtitle='Considerando todas las sucursales.'
)
fig.update_layout(
    yaxis=dict(
        title='Ingresos (log)',
        showgrid=True,
        gridcolor='lightgrey'
    ),
    plot_bgcolor='white'
)
fig.write_image(r'../assets/cajas_ingresos_totales.png')
fig.show()

**Lo que se muestra en la gráfica:**

La presencia de valores atípicos en los los ingresos por transacción, destaca el ticket que facturó $360.

In [80]:
fig=px.box(
    data_frame=df,
    y='transaction_qty',
    title='Diagrama de caja del volumen de ventas',
    subtitle='Considerando todas las sucursales.'
)
fig.update_layout(
    yaxis=dict(
        title='Cantidad de transacciones',
        gridcolor='lightgrey'
    ),
    plot_bgcolor='white'
)
fig.write_image(r'../assets/cajas_volumen_total.png')
fig.show()

**Lo que se muestra en la gráfica:**

Tres valores atípicos en la cantidad de productos facturados en un solo ticket.

## Visión general del menú

In [24]:
# obtenemos el menú general
menu=df[['product_detail','unit_price', 'store_location']].drop_duplicates()

In [25]:
menu.describe()

,unit_price
count,259.000000
mean,6.821853
std,7.378084
min,0.800000
25%,3.000000
50%,3.500000
75%,8.950000
max,45.000000


In [26]:
fig=px.box(
    data_frame=menu,
    y='unit_price',
    title='Diagrama de caja de precios de productos.',
    subtitle='Considerando todas las sucursales.'
)
fig.update_layout(
    yaxis=dict(
        title='Precio unitario [$]'
    )
)
fig.show()

### ¿Se ofrecen los mismos productos en todas las tiendas?

In [27]:
menu['store_location'].unique()

array(['Lower Manhattan', "Hell's Kitchen", 'Astoria'], dtype=object)

In [28]:
sucursales=menu['store_location'].unique()
menu_por_sucursal={}

for i in sucursales:
    menu_por_sucursal[i]=set(menu[menu['store_location']==i]['product_detail'].unique())

# verificamos si los menús son iguales
from itertools import combinations

for suc1, suc2 in combinations(sucursales, 2):
    diferentes=[]
    if menu_por_sucursal[suc1]==menu_por_sucursal[suc2]:
        print(f'{suc1} vs. {suc2}: IGUALES')
        
    else:
        print(f'{suc1} vs. {suc2}: DIFERENTES')
        diferentes.append(menu_por_sucursal[suc1]-menu_por_sucursal[suc2])
        print(diferentes)


Lower Manhattan vs. Hell's Kitchen: IGUALES
Lower Manhattan vs. Astoria: DIFERENTES
[{'Ouro Brasileiro shot'}]
Hell's Kitchen vs. Astoria: DIFERENTES
[{'Ouro Brasileiro shot'}]


La sucursal de Astoria no ofrece el producto *Ouro Brasileiro shot*.

## Análisis por categoría y producto

### ¿Qué genera mayores ingresos?

#### Categorías

In [81]:
ingresos_por_categoria=df.groupby('product_category')['total'].sum().sort_values(ascending=True).reset_index()
fig=px.pie(
    data_frame=ingresos_por_categoria,
    values='total',
    names='product_category',
    title='Ingresos totales por categoría',
    subtitle='Este gráfico considera todas las sucursales durante todo el lapso de tiempo.'
)
fig.update_layout(
    margin=dict(l=10, r=10, t=100, b=20),
    legend=dict(x=0.86, y=0.5)
)

fig.update_traces(
    textposition='inside',
    textinfo='percent+label'
    
)
fig.write_image(r'../assets/distribucion_global_ingresos.png')
fig.show()

**Lo que se muestra en la gráfica:**

*Coffee* es la categoría dominante, con el 38.6% de los ingresos de todas las sucursales, le siguen *Tea* con 28.1%  y *Bakery* con 11.8%. Por otro lado, los ingresos de *Packaged Chocolate*, *Flavours* y *Loose Tea* son marginales, lo que suguiere que una modificación al menú sobre estos tipos de producto, no resultaría en un cambio significativo.

#### Tipo de producto

In [83]:
# ingresos por tipo de producto
ingresos_por_tipo=df.groupby(
    by='product_type'
)['total'].sum().sort_values(ascending=True).reset_index()

fig=px.pie(
    data_frame=ingresos_por_tipo,
    values='total',
    names='product_type',
    title='Ingresos por tipo de producto',
    subtitle='Este gráfico considera todas las sucursales durante todo el lapso de tiempo.'
)

fig.update_layout(
    margin=dict(l=10, r=10, t=50, b=20),
    legend=dict(x=0.9, y=0.5)
)
fig.write_image(r'../assets/ingresos_globales_productos.png')
fig.show()


**Lo que se muestra en la gráfica:**

Los productos del tipo *Barista Espresso* son dominantes en la generación de ingresos, con 13.1%, le siguen los *Brewed Chai tea* (11%) y *Hot chocolate* (10.4%). En el opuesto se encuentran *Green beans* con apenas 0.19%, seguido por *Green tea* (0.21%) y *Organic chocolate* con 0.24%.

#### Producto específico

In [31]:
# ingresos por producto
ingresos_por_producto=df.groupby(['product_id', 'product_detail', 'product_type', 'product_category'])['total'].sum().sort_values(ascending=False).reset_index()
ingresos_por_producto

,product_id,product_detail,product_type,product_category,total
0,61,Sustainably Grown Organic Lg,Hot chocolate,Drinking Chocolate,21151.75
1,59,Dark chocolate Lg,Hot chocolate,Drinking Chocolate,21006.00
2,39,Latte Rg,Barista Espresso,Coffee,19112.25
3,41,Cappuccino Lg,Barista Espresso,Coffee,17641.75
4,55,Morning Sunrise Chai Lg,Brewed Chai tea,Tea,17384.00
...,...,...,...,...,...
75,11,Lemon Grass,Herbal tea,Loose Tea,1360.40
76,10,Guatemalan Sustainably Grown,Green beans,Coffee beans,1340.00
77,18,Spicy Eye Opener Chai,Chai tea,Loose Tea,1335.90
78,14,Earl Grey,Black tea,Loose Tea,1270.90


En el top 3 productos que más ingresos reportaron son:

1. Sustainably Grown Organic Lg (Hot chocolate)
2. Dark chocolate Lg (Hot chocolate)
3. Latte Rg (Barista Espresso)

In [84]:
fig=px.treemap(
    data_frame=ingresos_por_producto,
    path=[px.Constant('Ingresos totales'), 'product_category', 'product_type', 'product_detail'],
    values='total',
    title='Ingresos netos por producto'
)

fig.update_traces(
    textinfo='label+value+percent parent',
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Ventas: $%{value:,.0f}<br>"
        "% del total: %{percentRoot:.0%}"
        "<extra></extra>"  # oculta el nombre de la traza
    )
)
fig.write_image(r'../assets/ingresos_netos_productos_treemap.png')
fig.show()

**Lo que se muestra en el gráfico:**

En general, las tres categorías que representan gran parte de los ingresos totales son *Coffee*, *Tea* y *Bakery* con 39%, 28% y 12% respectivamente, al analizar cada una de ellas, se observa que:
- Dentro de las bebidas con café, las que son del tipo *Barista espresso* son las principales fuentes de ingresos.
  - En concreto, el *Latte Rg* (tamaño mediano) es la bebida principal con un 21%.
  - Le sigue el *Cappuccino Lg* (tamaño grande) quien aporta el 19%.
- En el partado de los tés, los tipo *Brewed Chai tea* son los principales.
  - Especialmente el Morning Sunrise Chai Lg, con un 23% de las ventas en ese tipo, esto lo convierte en la bebida a base de té que reporta más ingresos.
- La panadería se posiciona en el tercer puesto, siendo los del tipo *Scone* los panes que más ingresos aprotan.
  - El *Scottish Cream scone* se posiciona como el pan con mejores ingresos, con un 24% dentro de este tipo.
  - Le sigue el *Ginger scone* con 22%.

El producto que reportó más ingresos a las cafeterías fue el chocolate caliente *Sustainably Organic Lg* con más del 3% de las ventas totales. Es importante resaltar que solo cuatro bebidas de chocolate son las que acumulan más del 10% de los ingresos, esto habla sobre las preferencias de los clientes y realmente donde se puede encontrar un producto ancla.

In [85]:
fig=px.bar(
    data_frame=ingresos_por_producto.head(10),
    y='product_detail',
    x='total',
    title='Comparación los diez productos que más ingresos reportaron.',
    color='product_category',
    category_orders={
        'product_detail':ingresos_por_producto.head(10)['product_detail'].to_list()
        }
)
fig.update_layout(
    xaxis=dict(
        title='Ingresos netos [$]',
        showgrid=True
    ),
    yaxis=dict(
        title='Productos'
    ),
    showlegend=True,
    legend=dict(
        orientation='h',
        y=-0.3

    )
)
fig.write_image(r'../assets/top_10_ingresos.png')
fig.show()

In [34]:
# función para calcular la desviación estándar y la media de una muestra de productos
def medidas(dataframe, range, column):
    dataframe_sliced=dataframe.head(range)[column]
    std=dataframe_sliced.std()
    mean=dataframe_sliced.mean()
    return {'Desviación estándar':std, 'Promedio':mean, 'Coeficiente de variación (CV [%])':(std/mean)*100}
# para los diez primeros productos
medidas(ingresos_por_producto, 10, 'total')

{'Desviación estándar': 2064.7323018910274,
 'Promedio': 17737.525,
 'Coeficiente de variación (CV [%])': 11.640475781660786}

**Lo que muestra la gráfica:**

La distribución de ingresos para los diez primeros productos que más ingresos reportaron, destacan dos bebidas de chocolate: *Sustainably Grown Organic Lg* y *Dark Chocolate Lg*. Los ingresos promedio de esta muestra son de aproximadamente $17,737.5, con una desviación de la media de $2,064.7 lo cual representa un 11.64% respecto al promedio, esto indica una dispersión moderadamente homogénea.

## ¿Quiénes generan mayor volumen de ventas?
En esta sección analizaremos el volumen de ventas, o númeo de transacciones para cada categroría, tipo y producto de las tres cafeterías.

In [35]:
# volumen de ventas por producto
volumen_productos=df.groupby(['product_id', 'product_detail', 'product_type', 'product_category'])['transaction_qty'].sum().sort_values(ascending=False).reset_index()

volumen_productos

,product_id,product_detail,product_type,product_category,transaction_qty
0,50,Earl Grey Rg,Brewed Black tea,Tea,4708
1,59,Dark chocolate Lg,Hot chocolate,Drinking Chocolate,4668
2,54,Morning Sunrise Chai Rg,Brewed Chai tea,Tea,4643
3,38,Latte,Barista Espresso,Coffee,4602
4,44,Peppermint Rg,Brewed herbal tea,Tea,4564
...,...,...,...,...,...
75,7,Jamacian Coffee River,Premium Beans,Coffee beans,146
76,14,Earl Grey,Black tea,Loose Tea,142
77,10,Guatemalan Sustainably Grown,Green beans,Coffee beans,134
78,18,Spicy Eye Opener Chai,Chai tea,Loose Tea,122


### Top 10 productos por volumen de ventas

In [86]:
fig=px.bar(
    data_frame=volumen_productos.head(10),
    x='transaction_qty',
    y='product_detail',
    color='product_category',
    category_orders={
        'product_detail':volumen_productos.head(10)['product_detail'].to_list()
    },
    title='Comparación del volumen de venta de los diez primeros productos'
)
fig.update_layout(
    xaxis=dict(
        title='Volumen de ventas',
        showgrid=True,
        gridcolor='lightgrey'
    ),
    yaxis=dict(
        title='Producto',
    ),
    legend=dict(
        orientation='h',
        y=-0.3
    ),
    plot_bgcolor='white'
)
fig.write_image(f'../assets/top_10_volumen.png')
fig.show()

In [37]:
# medidas locales para los 10 productos más vendidos
medidas(volumen_productos, 10, 'transaction_qty')

{'Desviación estándar': 81.85597107114421,
 'Promedio': 4570.2,
 'Coeficiente de variación (CV [%])': 1.7910807201248133}

**Lo que se muestra en la gráfica:**

Tomando en cuenta los diez produtos más vendidos, se puede observar que el volumen promedio de ventas es de aproximadamente 4,570 unidades, con una desviación estándar de casi 82 unidades, es decir, cada uno de estos productos tiene una diferencia de aproximandamente 1.8% respecto al promedio. Esto nos dice que esta muestra de productos es homogénea y consistente entre los próductos líderes Se puede decir que, al menos para este conjunto, el produto más vendido (*Early Grey Rg*) no parece destacar cuantitativamente sobre los demás, es decir no es un valor atípico que distorcione la media.

In [38]:
fig=px.treemap(
    data_frame=volumen_productos,
    path=[px.Constant('Todo'), 'product_category', 'product_type', 'product_detail'],
    values='transaction_qty',
    title='Volumen de ventas por producto',
    subtitle='Cantidad total de productos vendidos agrupados por categoría y tipo'
)
fig.update_traces(
    textinfo='label+value+percent parent',
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Volumen: %{value:,.0f}<br>"
        "% del total: %{percentRoot:%}"
        "<extra></extra>"  # oculta el nombre de la traza
    )
)
fig.show()

**Lo que se muestra en la gráfica:**

En términos del volumen de ventas, tenemos a los tres productos más vendidos:
- *Earl grey Rg* (té)
- *Dark Chocolate Lg* (chocolate caliente)
- *Morning Sunrise Chai Rg* (té)

## Ingresos vs. volumen
Es interesante comparar los ingresos de los productos más vendidos y el volumen de ventas de los productos que más ingresos reportan a las cafeterías. En las siguientes gráficas se hará dicha comparación.

In [39]:
# Creamos una tabla con los ingresos totales y el volumen vendido para cada producto
ingresos_volumen=ingresos_por_producto.merge(
    volumen_productos.drop(columns=['product_type','product_category', 'product_detail']),
    how='left',
    on=['product_id']
)

ingresos_volumen=ingresos_volumen.rename(columns={'total':'ingresos_totales','transaction_qty':'volumen_total'})

### ¿Los productos con más ingresos son quienes tienen mejor volumen?

In [87]:
# graficamos ingresos vs volumen del top 10 en ingresos
ingresos_volumen_sort_ingresos=ingresos_volumen.sort_values(by='ingresos_totales', ascending=False).head(10)


fig=make_subplots(rows=1, cols=3, subplot_titles=('Ingresos', ' ', 'Volumen'))

fig.add_trace(go.Bar(
    x=ingresos_volumen_sort_ingresos['ingresos_totales'],
    y=ingresos_volumen_sort_ingresos['product_detail'],
    orientation='h'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=ingresos_volumen_sort_ingresos['volumen_total'],
    y=ingresos_volumen_sort_ingresos['product_detail'],
    orientation='h'
),row=1, col=3)

fig.update_layout(
    title=dict(
        text='Top 10 productos por ingresos',
        subtitle=dict(
            text='Comparativa entre los productos con mejores ingresos reportados y sus respectivos volúmenes de venta.'
        )
    ),
    # nombre de los ejes
    xaxis=dict(title="Ingresos totales"),
    yaxis=dict(title='Producto'),
    xaxis3=dict(title='Volumen total'),
    showlegend=False
)
fig.write_image(f'../assets/ingresos_vs_volumen.png')
fig.show()

In [41]:
medidas(ingresos_volumen_sort_ingresos, 10, 'ingresos_totales')

{'Desviación estándar': 2064.7323018910274,
 'Promedio': 17737.525,
 'Coeficiente de variación (CV [%])': 11.640475781660786}

**Lo que se muestra en la gráfica:**

Al analizar los diez primeros productos que aportan mayores ingresos a las cafeterías y compararlos con el volumen vendido para los respectivos artículos, se puede observar que, estrictamente, el producto con mejores ingresos no fue el que vendió más por volúmen. La la variación del volumen vendido respecto al promedio en esta muestra ronda el 11.6%, lo que indica una dispersión poco pronunciada.

### ¿ Los productos con mejor volumen de ventas son los de mejores ingresos?

In [88]:
ingresos_volumen_sort_volumen=ingresos_volumen.sort_values(by='volumen_total', ascending=False).head(10)

fig=make_subplots(rows=1, cols=3, subplot_titles=('Volumen', ' ', 'Ingresos'))

fig.add_trace(go.Bar(
    x=ingresos_volumen_sort_volumen['volumen_total'],
    y=ingresos_volumen_sort_volumen['product_detail'],
    orientation='h'
),row=1, col=1)

fig.add_trace(go.Bar(
    x=ingresos_volumen_sort_volumen['ingresos_totales'],
    y=ingresos_volumen_sort_volumen['product_detail'],
    orientation='h'
), row=1, col=3)

fig.update_layout(
    title=dict(
        text='Top 10 productos por volumen',
        subtitle=dict(
            text='Comparativa de los diez productos con mayor volumen de ventas y sus respectivos ingresos.'
        )
    ),
    showlegend=False,
    xaxis=dict(title='Volumen total'),
    xaxis3=dict(title='Ingresos totales'),
    yaxis=dict(title='Producto')
)
fig.write_image(f'../assets/volumen_vs_ingresos.png')
fig.show()

In [43]:
medidas(ingresos_volumen_sort_volumen, 10, 'ingresos_totales')

{'Desviación estándar': 4058.437061046065,
 'Promedio': 13497.125,
 'Coeficiente de variación (CV [%])': 30.06890031059255}

**Lo que se muestra en la gráfica:**

Cuando se comparan ahora los diez productos más vendidos respecto los ingresos que generan, el coeficiente de variación ronda el 30%, lo que indica ingresos más dispersos respecto a la media; se puede apreciar claramente en la gráfica anterior donde el producto más vendido (*Earl Grey Rg*) no es precisamente quien genera los mayores ingresos entre los diez, este puesto lo ocupa el segundo producto que más vendido: *Darl chocolate Lg*. Es interesante notar que el séptimo producto más vendido: *Latte Rg*, es el segundo que mejores ingresos genera.

## Análisis por sucursal

Se cuentan con las siguientes sucursales:

In [44]:
print(df[['store_location', 'store_id']].value_counts())

store_location   store_id
Hell's Kitchen   8           50735
Astoria          3           50599
Lower Manhattan  5           47782
Name: count, dtype: int64


### Ingresos por sucursal.

In [45]:
resumen_sucursales=df.groupby(['store_id', 'store_location']).agg(
    ingresos_totales=('total', 'sum'),
    volumen_total=('transaction_qty', 'sum'),
    promedio_por_transaccion=('total', 'mean')
).reset_index()

In [46]:
fig=px.bar(
    data_frame=resumen_sucursales.sort_values(by='ingresos_totales', ascending=True).reset_index(),
    y='store_location',
    x='ingresos_totales',
    orientation='h',
    title="Ingresos totales por sucursal"
)
fig.update_layout(
    xaxis=dict(
        title="Ingresos [$]",
        showgrid=False
    ),
    yaxis=dict(
        title="Sucursal",
        showgrid=False
    ),
    plot_bgcolor="rgba(0,0,0,0)"
    
)

fig.show()

**Lo que se muestra en la gráfica:**

En la gráfica anterior se puede observar los ingresos totales percibidos para cada sucursal, no se aprecia una dispersión considerable, por lo tanto se puede concluir que cada sucursal recibió aproximadamente los mismos ingresos.

### Transacciones por sucursal

In [47]:
fig=px.bar(
    data_frame=resumen_sucursales.sort_values(by='volumen_total', ascending=True).reset_index(),
    x='volumen_total',
    y='store_location',
    title='Unidades vendidas por sucursal'
)
fig.update_layout(
    xaxis=dict(
        title='Ventas unitarias',
        showgrid=False
    ),
    yaxis=dict(
        title='Sucursal',
        showgrid=False
    ),
    plot_bgcolor='rgba(0,0,0,0)'
)
fig.show()

**Lo que se muestra en la gráfica:**

Al igual que con los ingresos, estos muestran ventas unitarias entre sucurusales sin dispersión considereble, llegando a ser casi homogénea.

### Ingresos promedio por sucursal

In [48]:
resumen_sucursales.columns

Index(['store_id', 'store_location', 'ingresos_totales', 'volumen_total',
       'promedio_por_transaccion'],
      dtype='object')

In [89]:
fig=px.bar(
    data_frame=resumen_sucursales.sort_values(by='promedio_por_transaccion').reset_index(),
    x='store_location',
    y='promedio_por_transaccion',
    title='Transacción promedio por sucursal'
)
fig.update_layout(
    xaxis=dict(
        title='Sucursal'
    ),
    yaxis=dict(
        title='Transacción promedio [$]',
        range=[4,5]
    ),
    plot_bgcolor='white'
)
fig.write_image(r'../assets/transaccion_promedio_por_sucursal.png')
fig.show()

**Lo que se muestra en la gráfica:**

No hay una diferencia notable entre la transacción promedio entre sucursales.

### Ingresos por categoría en cada sucursal
Es posible que en cada sucursal se hayan vendido con mayor frecuencia ciertos productos, esto es importante para conocer las preferencias de los clientes que habitan cada zona y poder llevar a cabo estrategias que maximicen las ventas.

In [90]:
fig=px.bar(
    data_frame=df.groupby(['store_location', 'product_category'])['total'].sum().reset_index().sort_values(by='total', ascending=True),
    x='product_category',
    y='total',
    color='store_location',
    barmode='group',
    hover_data=['total'],
    title='Ingresos por categoría de producto en cada sucursal'
)
fig.update_layout(
    xaxis=dict(
        title='Categoría de producto',
        showgrid=False
    ),
    yaxis=dict(
        title='Ingresos [$]'
    ),
    plot_bgcolor='white',
    legend_title='Sucursal'
)
fig.write_image(r'../assets/ingresos_categoria_sucursal.png')
fig.show()

Resorting to unclean kill browser.


**Lo que se muestra en la gráfica:**

- *Coffee* es la categoría dominante en cuanto a ingresos en las tres sucursales, no se aprecia una diferencia considerable entre los ingresos reportados para cada sucursal.

- En el caso de *Tea* se aprecia una ligera diferencia a favor de la sucrusal ubicada en Astoria.
- No se aprecia una diferencia considerable entre los ingresos.
- La sucursal *Hell's Kitchen* recibió mejores ingresos por los productos de *Coffee beans* respecto a las demás.
- Los productos *Branded* tuvieron mejores ingresos en *Astoria* y *Lower Mahattan*.

### ¿Qué tipo de producto tiene mejores ingresos?

In [91]:
fig=px.bar(
    data_frame=df.groupby(['store_location', 'product_type'])['total'].sum().reset_index().sort_values(by='total', ascending=True),
    y='total',
    x='product_type',
    color='store_location',
    barmode='group',
    title='Ingresos por tipo de producto por sucursal'
)
fig.update_layout(
    xaxis=dict(
        title='Tipo de producto'
    ),
    yaxis=dict(
        title='Ingresos [$]',
        gridcolor='lightgray'
    ),
    plot_bgcolor='rgba(0,0,0,0)',
    legend=dict(
        title='Sucursal'
    )
)
fig.write_image(r'../assets/ingresos_tipo_producto_por_sucursal.png')
fig.show()

**Lo que se muestra en la gráfica**

Ingresos de cada sucursal agrupados por tipo de producto. Analizando los tres tipos de productos con mayores ingresos, el *Barista Espresso* reportó más flujo de efectivo en la sucursal *Hell's Kithcen*, seguido, sin notable diferencia, por *Lower Manhatttan*, mientras que en la tienda de *Astoria* se tuvo un flujo significativamente menor.

El *Brewed Chai tea* reportó mejores ingresos en la sucursal de *Astoria*, en *Hell's Kitchen* se tuvo un menor flujo aunque no de manera significativa, mientras tanto, *Lower Manhattan* reporta de una manera moderada menores ingresos respecto a *Astoria*.

El caso del *Gourmet Brewed coffee* no se nota una diferencia menos significativa entre las tres sucursales, llegando a ser casi homogéneo la captación de ingresos.

Sobre los demás tipos de producto, es sobresaliente el caso de *Premium Beans*, donde *Hell's Kitchen* obtuvo mayores ingresos sobre las otras dos sucursales.

### Volumen de ventas por categoría

In [52]:
fig=px.bar(
    data_frame=df.groupby(['store_location', 'product_category'])['transaction_qty'].sum().reset_index().sort_values(by='transaction_qty', ascending=True),
    x='product_category', 
    y='transaction_qty',
    color='store_location',
    barmode='group',
    title='Unidades vendidas de cada categoría agrupado por sucursal'
)
fig.update_layout(
    xaxis=dict(
        title='Categoría'
    ),
    yaxis=dict(
        title='Unidades vendidas',
        gridcolor='lightgrey'
    ),
    plot_bgcolor='rgba(0,0,0,0)',
    legend=dict(
        title='Sucursal'
    )
)
fig.show()

**Lo que se muestra en la gráfica**

No se aprecia una diferencia considerable entre los volúmenes de venta de la categoría dominante *Coffee* Hay un mejor constraste del número de transacciones en la categoría *Flavours*, donde *Lower Manhattan* fue quien tuvo mejor desempeño.

### Volumen de ventas por tipo de producto

In [53]:
fig=px.bar(
    data_frame=df.groupby(['store_location', 'product_type'])['transaction_qty'].sum().reset_index().sort_values(by='transaction_qty'),
    x='product_type',
    y='transaction_qty',
    color='store_location',
    barmode='group',
    title='Unidades vendidas de cada tipo de producto agrupado por sucursal'
)
fig.update_layout(
    xaxis=dict(
        title='Sucursal'
    ),
    yaxis=dict(
        title='Volumen de ventas',
        gridcolor='lightgray'
    ),
    plot_bgcolor='rgba(0,0,0,0)',
    legend=dict(
        title='Sucursal'
    )
)
fig.show()

**Lo que se muestra en la gráfica:**

Los productos tipo *Barista Espresso* no tienen un buen desempeño en la sucursal de *Astoria*. Por otro lado, en esa misma sucursal, los prodcutos del tipo *Brewed Chai tea* se vendieron más que en las demás sucursales.

Los clientes que frecuentan la sucursal de *Astoria* no son adeptos a los productos endulzantes, como se deja ver en *Regular syrup*, incluso con aquellos que no contienen azúcar directamente como *Sugar free syrup*.

## Análisis temporal

In [54]:
# para ordenar las gráficas
orden_meses=['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']

orden_dias=['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']

nuevos_titulos = {
    'Astoria': 'Astoria',
    "Hell's Kitchen": "Hell's Kitchen",
    'Lower Manhattan': 'Lower Manhattan'
}

### Diario

In [55]:
diario=df.groupby(['day', 'month_name', 'store_location']).agg(
    ingresos_totales=('total', 'sum'),
    ingreso_promedio=('total', 'mean'),
    ventas_unitarias=('transaction_qty', 'sum')
).reset_index()

In [92]:
diario_total = df.groupby(['day', 'month_name']).agg(
    ingresos_totales=('total', 'sum'),
    ingreso_promedio=('total', 'mean')
).reset_index()
# ingresos totales de las tres sucursales
fig=px.line(
    data_frame=diario_total,
    x='day',
    y='ingresos_totales',
    color='month_name',
    category_orders={'month_name': orden_meses},
    title='Histórico de ingresos totales',
    subtitle='Considerando todas las sucursales',
    markers=True
)
fig.update_layout(
    xaxis=dict(
        title='Día',
        showgrid=True,
        gridcolor='lightgrey',
        dtick='1D',
        range=[1,31]
    ),
    yaxis=dict(
        title='Ingresos totales [$]',
        showgrid=True,
        gridcolor='lightgrey'
    ),
    plot_bgcolor='white',
    legend=dict(
        title='Mes'
    )
)
fig.write_image(r'../assets/historico_ingresos.png')
fig.show()

**Lo que se muestra en la gráfica:**
- Enero y febrero son los meses donde se registraron los menores ingresos en las tres sucursales.
- Por otro lado, mayo y junio fueron los mejores meses en ingresos para las tres cafeterías.
- Se observa una tendencia positiva en los ingresos para los meses de marzo, abril, mayo y junio a partir del día 7 de cada mes.
- Se observa una caida conjunta pero breve para todos los meses a partir del día 28.

In [94]:
fig=px.line(
    data_frame=diario_total,
    x='day',
    y='ingreso_promedio',
    color='month_name',
    category_orders={'month_name': orden_meses},
    title='Ingreso promedio',
    subtitle='Considerando todas las sucursales',
    markers=True
)
fig.update_layout(
    xaxis=dict(
        title='Día',
        showgrid=True,
        gridcolor='lightgrey',
        dtick='1D',
        range=[1,31]
    ),
    yaxis=dict(
        title='Ingreso promedio [$]',
        showgrid=True,
        gridcolor='lightgrey'
    ),
    plot_bgcolor='white',
    legend=dict(
        title='Mes'
    )
)
fig.write_image(r'../assets/historico_ingreso_promedio.png')
fig.show()

**Lo que se muestra en la gráfica:**
- Enero, abril, mayo y junio muestran un *pico* de ingreso promedio entre los días 16 y 18 de cada mes, siendo enero el máximo con un promedio de $6.1.
- Por otra parte, febrero en el mismo intervalo tiene la disminución más pronunciada de todos los meses.
- En el histórico de ingresos enero es quien reporta menor entrada de efectivo, pero en promedio parece estar junto a los demás meses, destacando por el 17 de enero el cual es el día con mejor facturación promedio.
- Marzo es el único mes que no presenta el pico de facturación media.
- Todos los meses parecen muestrar una tendencia negativa breve hacia el día 28.para luego remontar.

#### Ingreso promedio por sucursal

In [95]:
fig=px.bar(
    data_frame=diario,
    x='day',
    y='ingreso_promedio',
    color='month_name',
    facet_row='store_location',
    barmode='stack',
    category_orders={'month_name': orden_meses},
    title='Ingreso promedio por sucursal'
)
fig.update_yaxes(title='')
# rotamos las anotaciones
fig.for_each_annotation(lambda a: a.update(textangle=0))
fig.for_each_annotation(lambda a: a.update(text=nuevos_titulos.get(a.text.split("=")[-1], a.text)))
fig.update_xaxes(
    dtick='1D'
)

fig.update_layout(
    plot_bgcolor='white',
    yaxis=dict(
        gridcolor='lightgrey'
    ),
    legend=dict(
        orientation='h',
        title='Mes',
        y=-0.2
    ),
    yaxis_title='Ingreso promedio',
    xaxis_title='Día'
)
fig.write_image(r'../assets/historico_ingreso_promedio_sucursal.png')
fig.show()

**Lo que se muestra en la gráfica:**

En el análisis del promedio global se mostró que el día 17 muestra un aumento considerable en los ingresos, al desglosar por sucursal se nota que la contribución a este aumento viene de la sucursal *Hell's Kitchen* quien el día 17 de enero, abril, mayo y junio reportaron un aumento significativo en la facturación promedio.

En general, las tres sucursales tienen una facturación promedio relativamente homogénea a lo largo del mes.

#### Histórico de ingresos por sucursal

In [59]:
fig=px.line(
    data_frame=diario,
    x='day',
    y='ingresos_totales',
    color='month_name',
    facet_row='store_location',
    facet_row_spacing=0.1,
    markers=True,
    title='Histórico de ingresos diarios por sucursal',
    category_orders={'month_name': orden_meses}
)

fig.update_xaxes(
    dtick='1D',
    gridcolor='lightgrey',
    title_text=''
)

fig.update_yaxes(
    showgrid=True,
    gridcolor='lightgrey',
    title_text=''
)

fig.update_layout(
    plot_bgcolor='white',
    legend_title='Mes',
    legend=dict(
        x=0.1,
        y=-0.2,
        orientation='h'
    ),
    xaxis_title='Día',
    yaxis_title='Ingresos [$]'
)
# rotar los títulos de cada gráfica
fig.for_each_annotation(lambda a: a.update(textangle=0))



fig.for_each_annotation(lambda a: a.update(text=nuevos_titulos.get(a.text.split("=")[-1], a.text)))

fig.show()

#### Histórico de volumen de ventas por sucursal

In [60]:
fig=px.line(
    data_frame=diario,
    x='day',
    y='ventas_unitarias',
    color='month_name',
    facet_row='store_location',
    facet_row_spacing=0.1,
    markers=True,
    title='Histórico de volumen de ventas totales diarios por sucursal',
    category_orders={'month_name': orden_meses}
)

fig.update_xaxes(
    dtick='1D',
    gridcolor='lightgrey',
    title_text=''
)

fig.update_yaxes(
    showgrid=True,
    gridcolor='lightgrey',
    title_text=''
)

fig.update_layout(
    plot_bgcolor='white',
    legend_title='Mes',
    legend=dict(
        x=0.1,
        y=-0.2,
        orientation='h'
    ),
    xaxis_title='Día',
    yaxis_title='Productos vendidos'
)
# rotar los títulos de cada gráfica
fig.for_each_annotation(lambda a: a.update(textangle=0))
fig.for_each_annotation(lambda a: a.update(text=nuevos_titulos.get(a.text.split("=")[-1], a.text)))

fig.show()

## Por hora

### Totales

In [96]:
ingresos_totales_por_hora=df.groupby(['hour', 'day_name']).agg(
    ingresos_totales=('total', 'sum')
).reset_index()

fig=px.bar(
    data_frame=ingresos_totales_por_hora,
    x='hour',
    y='ingresos_totales',
    color='day_name',
    title='Ingresos totales por hora',
    subtitle='Considerando todas las sucursales',
    category_orders={'day_name': orden_dias}
)
fig.update_layout(
    xaxis=dict(
        title='Hora',
        gridcolor='lightgrey'
    ),
    yaxis=dict(
        title='Ingresos [$]',
        gridcolor='lightgrey'
    ),
    plot_bgcolor='rgba(0,0,0,0)',
    legend_title='Dia de la semana'
)
fig.update_xaxes(
    dtick='1D'
)
fig.write_image(r'../assets/ingresos_por_hora.png')
fig.show()

**Lo que se muestra en la gráfica:**

La hora de apertura de las sucursales ocurre a las 6 hrs., de 7 a 10 hrs. ocurre el máximo de ingresos. Pasadas las 11 y hasta las 17 hrs los ingresos son homogéneos durante todos los días. La hora del cierre ocurre a las 20 hrs. y los ingresos sufren una caída considerable por debajo de los $5K.

### Por sucursal

In [62]:
por_hora=df.groupby(['hour', 'day_name', 'store_location']).agg(
    ingresos_totales=('total', 'sum'),
    ingreso_promedio=('total', 'mean')
).reset_index()

In [97]:
fig=px.bar(
    data_frame=por_hora,
    x='hour',
    y='ingresos_totales',
    color='day_name',
    title='Ingresos totales por sucursal a lo largo del día',
    category_orders={'day_name': orden_dias},
    facet_row='store_location'
)
fig.update_xaxes(
    title='',
    dtick='1D'
)
fig.update_yaxes(
    title='',
    showgrid=True,
    gridcolor='lightgrey'
)
fig.update_layout(
    xaxis=dict(
        title='Hora',
        gridcolor='lightgrey'
    ),
    yaxis=dict(
        title='Ingresos totales [$]',
        gridcolor='lightgrey'
    ),
    plot_bgcolor='rgba(0,0,0,0)',
    legend=dict(
        x=0.1,
        y=-0.2,
        orientation='h',
        title='Día de la semana'
    )
)
fig.for_each_annotation(lambda a: a.update(textangle=0))
fig.for_each_annotation(lambda a: a.update(text=nuevos_titulos.get(a.text.split("=")[-1], a.text)))
fig.write_image(r'../assets/ingresos_sucrusales_por_hora.png')
fig.show()

**Lo que se muestra en la gráfica:**

- La sucursal de Astoria tiene un horario de 7 a 20 hrs., en cambio, Hell´s Kitchen y Lower Manhattan abren de 6 a 20 hrs.
- Hell´s Kitchen tuvo un máximo de ingresos entre 8 y 10 hrs., de 12 a 19 hrs. los ingresos se mantienen estables a lo largo de los días. Llegando a las 20 hrs. la facturación total cayó por debajo de los $5K.
- Lower Manhattan tiene un máximo de ingresos más prologado, desde 6 hasta las 10 hrs. A partir de las 11 y hasta las 17 hrs los ingresos se mantienen estables para toda la semana. La caída abrupta de los ingresos ocurrió desde las 19 y hasta las 20 hrs. Esto puede indicar que dicha sucursal puede cerrar más temprano.
- Comparado con la otras dos sucursales, Astoria se comporta de manera más homogénea, desde su apertura se reportó un incremento en los ingresos hasta las 10 hrs. Pasado ese tiempo y hasta el cierre los ingresos se mantuvieron estables y homogéneos.

In [98]:
fig=px.bar(
    data_frame=por_hora,
    x='hour',
    y='ingreso_promedio',
    color='day_name',
    title='Ingreso promedio por sucursal a lo largo del día',
    category_orders={'day_name': orden_dias},
    facet_row='store_location'
)
fig.update_xaxes(
    title='',
    dtick='1D'
)
fig.update_yaxes(
    title='',
    showgrid=True,
    gridcolor='lightgrey'
)
fig.update_layout(
    xaxis=dict(
        title='Hora',
        gridcolor='lightgrey'
    ),
    yaxis=dict(
        title='Ingreso promedio[$]',
        gridcolor='lightgrey'
    ),
    plot_bgcolor='rgba(0,0,0,0)',
    legend=dict(
        x=0.1,
        y=-0.2,
        orientation='h',
        title='Día de la semana'
    )
)
fig.for_each_annotation(lambda a: a.update(textangle=0))
fig.for_each_annotation(lambda a: a.update(text=nuevos_titulos.get(a.text.split("=")[-1], a.text)))
fig.write_image(r'../assets/ingreso_promedio_por_hora.png')
fig.show()

**Lo que se muestra en la gráfica:**

En las tres sucursales se observa un ingreso promedio homogéneo a lo largo del día y durante la semana, con excepción de la sucursal Lower Manhattan donde para las 19 hrs. hay un aumento considerable en el ingreso promedio, especialmente en el día sábado.


## Por día de la semana

In [65]:
por_dia_dela_semana=df.groupby(['day_name', 'store_location']).agg(
    ingresos_totales=('total', 'sum'),
    ingreso_promedio=('total', 'mean')
).reset_index()

In [66]:
fig=px.bar(
    data_frame=por_dia_dela_semana,
    x='day_name',
    y='ingresos_totales',
    color='store_location',
    barmode='group',
    category_orders={'day_name': orden_dias},
    title='Histórico de ingresos totales por día de la semana'
)

fig.update_layout(
    plot_bgcolor='white',
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        title='Ingreso total [$]',
        range=[25000,38000]
    ),
    xaxis_title='',
    legend_title='Sucursal'
)


fig.show()

**Lo que muestra la gráfica:**

Los lunes Lower Manhattan reportó ingresos ligeramente mayores, los martes y sabados no son los días más favorables para la sucursal de Astoria. Martes, viernes y domingo son fueron los días que Hell's Kitchen reportó mejores ingresos.

In [100]:
fig=px.bar(
    data_frame=por_dia_dela_semana,
    x='day_name',
    y='ingreso_promedio',
    color='store_location',
    barmode='group',
    category_orders={'day_name': orden_dias},
    title='Ingreso promedio por día de la semana.'
)

fig.update_layout(
    plot_bgcolor='white',
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        title='Ingreso promedio [$]',
        range=[4,5]
    ),
    xaxis_title='',
    legend_title='Sucursal'
)

fig.write_image(r'../assets/ingreso_promedio_semanal.png')
fig.show()

**Lo que muestra la gráfica:**

En promedio, Lower Manhattan tiene una facturación mayor durante toda la semana, siendo el martes su mejor día.
